INTEGRATE THIS INTO DATAHUB.PY

In [2]:
import difflib
import pandas as pd
import espn_api.basketball as bb
from yfpy.query import YahooFantasySportsQuery
from nba_api.stats.static import players, teams
from nba_api.stats.library.parameters import Season
from sqlalchemy.dialects.postgresql.base import PGDialect; PGDialect._get_server_version_info = lambda *args: (9, 2)
from dataHub import dataHub
dh = dataHub()

db_con = dh.db_connect('postgre')
fty_con = dh.fty_con(db_con)

In [1]:
f_con = fty_con['ESPN;95537']

df = []
for player in f_con.player_map:
    if str(player).isdigit():
        df.append({
            'espn_id': player,
            'espn_name': f_con.player_map[player]
        })        

df_espn = pd.DataFrame(df).sort_values('espn_id')

In [ ]:
f_con = fty_con['Yahoo;121793']

df = []
for player in f_con.get_league_players():
    df.append({
        'yahoo_id': player.player_id ,
        'yahoo_name': player.name.full
    })

df_yahoo = pd.DataFrame(df)

In [ ]:
df = []
for player in players.get_active_players():
    df.append({
        'nba_id': player['id'],
        'nba_name': player['full_name']
    })

df_db = pd.DataFrame(df)

In [ ]:
espn_match = df_db['nba_name'].apply(lambda x: difflib.get_close_matches(x, df_espn['espn_name']))
df_db['espn_name'] = [el[0] if len(el) > 0 else None for el in espn_match]
df_db = df_db.merge(df_espn, on='espn_name', how='left')

yahoo_match = df_db['nba_name'].apply(lambda x: difflib.get_close_matches(x, df_yahoo['yahoo_name']))
df_db['yahoo_name'] = [el[0] if len(el) > 0 else None for el in yahoo_match]
df_db = df_db.merge(df_yahoo, on='yahoo_name', how='left')

df_db.to_clipboard(index=False)

# Then export to excel remove matches that aren't correct, then remove duplicates in nba_name column